# Coach DNA EDA

This notebook explores the cleaned offensive play universe and the first analytical layers used to build team profiles and league baselines.

## Goals
- confirm the cleaned table loads correctly
- inspect season coverage and row counts
- understand the distribution of plays by down, distance, field position, and score state
- review how run/dropback behavior changes across situations
- identify patterns worth carrying into team profiling and scoring

In [26]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

candidate_paths = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None

for path in candidate_paths:
    if (path / "python").exists() and (path / "data").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root from notebook.")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("OUTPUT_TABLES_DIR:", OUTPUT_TABLES_DIR)

PROJECT_ROOT: /Users/Tip/Desktop/ea-coach-dna-calibration
PROCESSED_DATA_DIR: /Users/Tip/Desktop/ea-coach-dna-calibration/data/processed
OUTPUT_TABLES_DIR: /Users/Tip/Desktop/ea-coach-dna-calibration/outputs/tables


In [27]:
team_baseline_features = pd.read_csv(
    PROCESSED_DATA_DIR / "team_baseline_features_2025_vs_2023_2025.csv"
)

coach_dna_situation_scores = pd.read_csv(
    PROCESSED_DATA_DIR / "coach_dna_situation_scores_2025.csv"
)

coach_dna_team_summary = pd.read_csv(
    PROCESSED_DATA_DIR / "coach_dna_team_summary_2025.csv"
)

print("team_baseline_features:", team_baseline_features.shape)
print("coach_dna_situation_scores:", coach_dna_situation_scores.shape)
print("coach_dna_team_summary:", coach_dna_team_summary.shape)

team_baseline_features: (671, 85)
coach_dna_situation_scores: (671, 62)
coach_dna_team_summary: (32, 13)


## 1. Basic Structure Checks
Start by confirming the exported feature and score tables have the expected size and shape.

In [28]:
team_baseline_features.head()

,profile_season,team,situation_order,situation_name,baseline_type,baseline_start_season,baseline_end_season,season_count,team_count,team_season_count,baseline_quality,team_play_count,team_sample_quality,sample_vs_baseline_context,team_meets_min_sample_50,team_meets_min_sample_20,meets_strong_team_sample,meets_usable_team_sample,dropback_rate_team,dropback_rate_baseline,dropback_rate_delta,dropback_rate_pct_delta,rush_rate_team,rush_rate_baseline,rush_rate_delta,rush_rate_pct_delta,pass_attempt_rate_team,pass_attempt_rate_baseline,pass_attempt_rate_delta,pass_attempt_rate_pct_delta,shotgun_rate_team,shotgun_rate_baseline,shotgun_rate_delta,shotgun_rate_pct_delta,no_huddle_rate_team,no_huddle_rate_baseline,no_huddle_rate_delta,no_huddle_rate_pct_delta,success_rate_team,success_rate_baseline,success_rate_delta,success_rate_pct_delta,first_down_rate_team,first_down_rate_baseline,first_down_rate_delta,first_down_rate_pct_delta,touchdown_rate_team,touchdown_rate_baseline,touchdown_rate_delta,touchdown_rate_pct_delta,turnover_rate_team,turnover_rate_baseline,turnover_rate_delta,turnover_rate_pct_delta,sack_rate_team,sack_rate_baseline,sack_rate_delta,sack_rate_pct_delta,explosive_play_rate_team,explosive_play_rate_baseline,explosive_play_rate_delta,explosive_play_rate_pct_delta,explosive_dropback_rate_team,explosive_dropback_rate_baseline,explosive_dropback_rate_delta,explosive_dropback_rate_pct_delta,explosive_run_rate_team,explosive_run_rate_baseline,explosive_run_rate_delta,explosive_run_rate_pct_delta,avg_yards_gained_team,avg_yards_gained_baseline,avg_yards_gained_delta,avg_yards_gained_pct_delta,avg_epa_team,avg_epa_baseline,avg_epa_delta,avg_epa_pct_delta,is_more_dropback_heavy,is_more_run_heavy,is_more_shotgun_heavy,is_more_no_huddle_heavy,is_more_efficient_epa,is_more_successful,is_more_explosive
0,2025,ARI,1,all_offense,league_multi_season,2023,2025,3,32,96,strong,1070,strong,stable_team_sample,1,1,1,1,0.6607,0.5737,0.0870,0.151647,0.3393,0.4263,-0.0870,-0.204082,0.6607,0.5737,0.0870,0.151647,0.7168,0.6967,0.0201,0.028850,0.1215,0.1039,0.0176,0.169394,0.4308,0.4345,-0.0037,-0.008516,0.2925,0.2932,-0.0007,-0.002387,0.0374,0.0402,-0.0028,-0.069652,0.0196,0.0274,-0.0078,-0.284672,0.0551,0.0401,0.0150,0.374065,0.1290,0.1328,-0.0038,-0.028614,0.0963,0.0838,0.0125,0.149165,0.0327,0.0490,-0.0163,-0.332653,5.1785,5.4305,-0.2520,-0.046405,-0.0169,-0.0054,-0.0115,-2.129630,1,0,1,1,0,0,0
1,2025,ARI,2,early_down,league_multi_season,2023,2025,3,32,96,strong,818,strong,stable_team_sample,1,1,1,1,0.6320,0.5246,0.1074,0.204727,0.3680,0.4754,-0.1074,-0.225915,0.6320,0.5246,0.1074,0.204727,0.6687,0.6436,0.0251,0.038999,0.1333,0.1086,0.0247,0.227440,0.4267,0.4339,-0.0072,-0.016594,0.2518,0.2564,-0.0046,-0.017941,0.0342,0.0363,-0.0021,-0.057851,0.0147,0.0244,-0.0097,-0.397541,0.0452,0.0296,0.0156,0.527027,0.1161,0.1311,-0.0150,-0.114416,0.0844,0.0774,0.0070,0.090439,0.0318,0.0538,-0.0220,-0.408922,5.0831,5.4816,-0.3985,-0.072698,0.0018,0.0014,0.0004,0.285714,1,0,1,1,1,0,0
2,2025,ARI,3,third_down,league_multi_season,2023,2025,3,32,96,strong,224,strong,stable_team_sample,1,1,1,1,0.7545,0.7494,0.0051,0.006805,0.2455,0.2506,-0.0051,-0.020351,0.7545,0.7494,0.0051,0.006805,0.8884,0.8953,-0.0069,-0.007707,0.0893,0.0875,0.0018,0.020571,0.4464,0.4242,0.0222,0.052334,0.4241,0.3997,0.0244,0.061046,0.0446,0.0487,-0.0041,-0.084189,0.0402,0.0359,0.0043,0.119777,0.0893,0.0778,0.0115,0.147815,0.1696,0.1404,0.0292,0.207977,0.1295,0.1070,0.0225,0.210280,0.0402,0.0333,0.0069,0.207207,5.5938,5.3482,0.2456,0.045922,-0.0652,-0.0507,-0.0145,-0.285996,1,0,0,1,0,1,1
3,2025,ARI,4,fourth_down,league_multi_season,2023,2025,3,32,96,good,28,thin,small_team_sample,0,1,0,0,0.7500,0.6225,0.1275,0.204819,0.2500,0.3775,-0.1275,-0.337748,0.7500,0.6225,0.1275,0.204819,0.7500,0.6750,0.0750,0.111111,0.0357,0.0943,-0.0586,-0.621421,0.4286,0.5443,-0.1157,-0.212567,0.4286,0.5475,-0.1189,-0.217169,0.0714,0.0926,-0.0212,-0.228942,0.0000,0.0492,-0.0492,-1.000000,0.0714,0.0504,0.0210,0.41

In [29]:
coach_dna_situation_scores.head()

,profile_season,team,situation_order,situation_name,team_play_count,team_sample_quality,sample_vs_baseline_context,sample_reliability_score,sample_multiplier,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,coach_dna_score_raw,coach_dna_score_adjusted,tendency_profile_label,tempo_profile_label,formation_profile_label,efficiency_profile_label,dropback_rate_team,dropback_rate_baseline,dropback_rate_delta,rush_rate_team,rush_rate_baseline,rush_rate_delta,pass_attempt_rate_team,pass_attempt_rate_baseline,pass_attempt_rate_delta,shotgun_rate_team,shotgun_rate_baseline,shotgun_rate_delta,no_huddle_rate_team,no_huddle_rate_baseline,no_huddle_rate_delta,avg_epa_team,avg_epa_baseline,avg_epa_delta,success_rate_team,success_rate_baseline,success_rate_delta,explosive_play_rate_team,explosive_play_rate_baseline,explosive_play_rate_delta,turnover_rate_team,turnover_rate_baseline,turnover_rate_delta,sack_rate_team,sack_rate_baseline,sack_rate_delta,dropback_rate_signal_score,shotgun_rate_signal_score,no_huddle_rate_signal_score,pass_attempt_rate_signal_score,avg_epa_score,success_rate_score,first_down_rate_score,touchdown_rate_score,explosive_play_rate_score,explosive_dropback_rate_score,explosive_run_rate_score,turnover_rate_score,sack_rate_score
0,2025,ARI,1,all_offense,1070,strong,stable_team_sample,100.0,1.00,57.81250,35.93750,45.833333,50.00000,51.875000,51.875000,more_dropback_heavy_than_baseline,close_to_baseline_tempo,close_to_baseline_shotgun_usage,close_to_baseline_efficiency,0.6607,0.5737,0.0870,0.3393,0.4263,-0.0870,0.6607,0.5737,0.0870,0.7168,0.6967,0.0201,0.1215,0.1039,0.0176,-0.0169,-0.0054,-0.0115,0.4308,0.4345,-0.0037,0.1290,0.1328,-0.0038,0.0196,0.0274,-0.0078,0.0551,0.0401,0.0150,96.875,9.375,28.125,96.875,31.250,34.3750,43.750,34.3750,46.875,84.3750,6.250,81.2500,18.750
1,2025,ARI,2,early_down,818,strong,stable_team_sample,100.0,1.00,60.93750,35.93750,34.375000,46.87500,51.250000,51.250000,more_dropback_heavy_than_baseline,faster_than_baseline,close_to_baseline_shotgun_usage,close_to_baseline_efficiency,0.6320,0.5246,0.1074,0.3680,0.4754,-0.1074,0.6320,0.5246,0.1074,0.6687,0.6436,0.0251,0.1333,0.1086,0.0247,0.0018,0.0014,0.0004,0.4267,0.4339,-0.0072,0.1161,0.1311,-0.0150,0.0147,0.0244,-0.0097,0.0452,0.0296,0.0156,100.000,9.375,34.375,100.000,34.375,31.2500,40.625,37.5000,28.125,71.8750,3.125,84.3750,9.375
2,2025,ARI,3,third_down,224,strong,stable_team_sample,100.0,1.00,11.71875,56.25000,81.250000,25.00000,39.023438,39.023438,close_to_baseline_run_pass_split,close_to_baseline_tempo,close_to_baseline_shotgun_usage,close_to_baseline_efficiency,0.7545,0.7494,0.0051,0.2455,0.2506,-0.0051,0.7545,0.7494,0.0051,0.8884,0.8953,-0.0069,0.0893,0.0875,0.0018,-0.0652,-0.0507,-0.0145,0.4464,0.4242,0.0222,0.1696,0.1404,0.0292,0.0402,0.0359,0.0043,0.0893,0.0778,0.0115,12.500,18.750,3.125,12.500,43.750,68.7500,65.625,46.8750,87.500,87.5000,68.750,21.8750,28.125
3,2025,ARI,4,fourth_down,28,thin,small_team_sample,55.0,0.75,60.93750,21.87500,67.708333,61.71875,51.968750,38.976562,more_dropback_heavy_than_baseline,slower_than_baseline,more_shotgun_than_baseline,less_efficient_than_baseline,0.7500,0.6225,0.1275,0.2500,0.3775,-0.1275,0.7500,0.6225,0.1275,0.7500,0.6750,0.0750,0.0357,0.0943,-0.0586,-0.1766,0.1649,-0.3415,0.4286,0.5443,-0.1157,0.1786,0.1209,0.0577,0.0000,0.0492,-0.0492,0.0714,0.0504,0.0210,75.000,43.750,50.000,75.000,28.125,10.9375,12.500,35.9375,84.375,90.6250,28.125,92.1875,31.250
4,2025,ARI,5,short_yardage,93,good,usable_team_sample,80.0,0.90,25.00000,36.71875,11.979167,51.56250,31.382812,28.244531,close_to_baseline_run_pass_split,close_to_baseline_tempo,close_to_baseline_shotgun_usage,less_efficient_than_baseline,0.3118,0.3476,-0.0358,0.6882,0.6524,0.0358,0.3118,0.3476,-0.0358,0.4839,0.5322,-0.0483,0.1828,0.1642,0.0186,-0.0485,0.0647,-0.1132,0.5591,0.5961,-0.0370,0.0430,0.0906,-0.0476,0.0108,0.0214,-0.0106,0.0215,0.0229,-0.0014,31.250,18.750,18.750,31.250,28.125,31.2500,53.125,34.

In [30]:
coach_dna_team_summary.head()

,profile_season,team,scored_situations,overall_coach_dna_score,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_situation,top_signal_score,lowest_signal_situation,lowest_signal_score
0,2025,LA,13,71.4506,62.3970,88.1404,70.5575,71.6600,96.3510,neutral_early_down,83.9844,two_minute_game,34.2305
1,2025,BUF,13,70.8384,72.2995,77.7592,70.5647,48.6145,96.0830,third_down,79.5703,fourth_down,45.8760
2,2025,WAS,13,68.2717,80.9874,56.2046,64.8656,45.0901,96.2908,tied_early_down,82.3633,two_minute_game,37.3359
3,2025,CIN,13,65.4883,74.2507,65.0623,41.7981,56.2964,97.3112,red_zone,82.0703,fourth_down,51.8379
4,2025,BAL,13,61.8666,70.4730,54.6368,68.3142,27.7396,95.3569,leading_early_down,70.8594,two_minute_game,22.9512


In [31]:
pd.Series({
    "feature_rows": len(team_baseline_features),
    "feature_teams": team_baseline_features["team"].nunique(),
    "feature_situations": team_baseline_features["situation_name"].nunique(),
    "situation_score_rows": len(coach_dna_situation_scores),
    "team_summary_rows": len(coach_dna_team_summary),
})

feature_rows            671
feature_teams            32
feature_situations       21
situation_score_rows    671
team_summary_rows        32
dtype: int64

## 2. Situation Coverage
These checks confirm that every team and every situation are represented the way we expect.

In [32]:
team_baseline_features.groupby("team").size().sort_values().head(10)

team
NYJ    20
ARI    21
TB     21
SF     21
SEA    21
PIT    21
PHI    21
NYG    21
NO     21
NE     21
dtype: int64

In [33]:
team_baseline_features.groupby("situation_name").size().sort_values()

situation_name
leading_two_plus_scores     31
all_offense                 32
trailing_two_plus_scores    32
trailing_one_score          32
trailing_early_down         32
trailing                    32
tied_early_down             32
tied                        32
third_down                  32
short_yardage               32
red_zone                    32
one_score                   32
neutral_early_down          32
leading_one_score           32
leading_early_down          32
leading                     32
goal_to_go                  32
fourth_down                 32
early_down                  32
two_minute_game             32
two_minute_half             32
dtype: int64

In [34]:
team_baseline_features.groupby("situation_name")["team_play_count"].agg(["count", "min", "median", "mean", "max"]).sort_values("mean")

,count,min,median,mean,max
situation_name,,,,,
fourth_down,32,12,28.0,27.562500,41
two_minute_game,32,34,45.5,45.812500,63
goal_to_go,32,37,63.0,62.875000,91
short_yardage,32,76,101.5,101.906250,129
two_minute_half,32,91,123.0,120.281250,143
leading_two_plus_scores,31,23,125.0,136.225806,292
tied_early_down,32,96,146.0,149.218750,209
red_zone,32,101,164.5,160.656250,203
tied,32,122,186.0,193.531250,270


## 3. Team Sample Size Review
This helps identify which situations are naturally large, small, or noisy.

In [35]:
sample_summary = (
    team_baseline_features
    .groupby("situation_name", as_index=False)
    .agg(
        avg_team_play_count=("team_play_count", "mean"),
        median_team_play_count=("team_play_count", "median"),
        min_team_play_count=("team_play_count", "min"),
        max_team_play_count=("team_play_count", "max"),
    )
    .sort_values("avg_team_play_count")
)

sample_summary

,situation_name,avg_team_play_count,median_team_play_count,min_team_play_count,max_team_play_count
2,fourth_down,27.562500,28.0,12,41
19,two_minute_game,45.812500,45.5,34,63
3,goal_to_go,62.875000,63.0,37,91
11,short_yardage,101.906250,101.5,76,129
20,two_minute_half,120.281250,123.0,91,143
7,leading_two_plus_scores,136.225806,125.0,23,292
14,tied_early_down,149.218750,146.0,96,209
10,red_zone,160.656250,164.5,101,203
13,tied,193.531250,186.0,122,270
6,leading_one_score,204.531250,209.0,63,322


In [36]:
team_baseline_features["team_sample_quality"].value_counts()

team_sample_quality
strong       538
good          70
thin          61
very_thin      2
Name: count, dtype: int64

In [37]:
team_baseline_features.groupby(["situation_name", "team_sample_quality"]).size().unstack(fill_value=0)

team_sample_quality,good,strong,thin,very_thin
situation_name,,,,
all_offense,0,32,0,0
early_down,0,32,0,0
fourth_down,0,0,30,2
goal_to_go,28,0,4,0
leading,1,31,0,0
leading_early_down,2,29,1,0
leading_one_score,3,29,0,0
leading_two_plus_scores,6,21,4,0
neutral_early_down,0,32,0,0


## 4. League Baseline Tendencies by Situation
This section shows how the league tends to behave across the major situations.

In [38]:
baseline_view = (
    team_baseline_features[
        [
            "situation_order",
            "situation_name",
            "dropback_rate_baseline",
            "rush_rate_baseline",
            "shotgun_rate_baseline",
            "no_huddle_rate_baseline",
            "avg_epa_baseline",
            "success_rate_baseline",
            "explosive_play_rate_baseline",
        ]
    ]
    .drop_duplicates()
    .sort_values("situation_order")
    .reset_index(drop=True)
)

baseline_view

,situation_order,situation_name,dropback_rate_baseline,rush_rate_baseline,shotgun_rate_baseline,no_huddle_rate_baseline,avg_epa_baseline,success_rate_baseline,explosive_play_rate_baseline
0,1,all_offense,0.5737,0.4263,0.6967,0.1039,-0.0054,0.4345,0.1328
1,2,early_down,0.5246,0.4754,0.6436,0.1086,0.0014,0.4339,0.1311
2,3,third_down,0.7494,0.2506,0.8953,0.0875,-0.0507,0.4242,0.1404
3,4,fourth_down,0.6225,0.3775,0.6750,0.0943,0.1649,0.5443,0.1209
4,5,short_yardage,0.3476,0.6524,0.5322,0.1642,0.0647,0.5961,0.0906
5,6,red_zone,0.5029,0.4971,0.6589,0.0945,-0.0022,0.4224,0.0512
6,7,goal_to_go,0.4757,0.5243,0.6110,0.0861,0.0097,0.4380,0.0074
7,8,two_minute_half,0.7349,0.2651,0.8778,0.1634,0.0083,0.4231,0.1271
8,9,two_minute_game,0.6818,0.3182,0.8173,0.1726,-0.0600,0.3942,0.1268
9,10,one_score,0.5650,0.4350,0.6859,0.0863,0.0035,0.4369,0.1332


### How to read this table

This table shows the **league baseline** for each major game situation. In other words, it tells us what "normal NFL behavior" looks like before we start layering in team-specific coaching DNA.

#### What the key numbers mean

- `dropback_rate_baseline`  
  This shows how often the average NFL offense leans into dropback behavior in that situation.  
  A **higher** number means the league expectation is more pass-oriented there.  
  For CPU tuning, this can act as the default behavioral anchor before adding team-specific adjustments.

- `rush_rate_baseline`  
  This shows how often the average offense leans into the run in that situation.  
  A **higher** number means the league norm is more run-oriented.  
  For CPU tuning, this helps define what a neutral or default run tendency should look like.

- `shotgun_rate_baseline`  
  This shows how often shotgun is the baseline structural choice.  
  For CPU logic, this can inform formation preference by situation.

- `no_huddle_rate_baseline`  
  This shows how often the league speeds up in that situation.  
  For CPU tuning, this is useful for tempo logic and late-game pace behavior.

- `avg_epa_baseline`  
  This tells us how productive the average offense is in that situation.  
  A **higher** number means the situation is more favorable or more efficiently handled league-wide.  
  For tuning, this helps distinguish whether a team-specific behavior is merely different or actually effective.

- `success_rate_baseline`  
  This shows how often the average offense produces a positive-EPA outcome in that situation.  
  For CPU tuning, it provides a benchmark for what "stable, on-schedule" behavior looks like.

#### Why this matters for CPU-controlled coaching decisions

A football game should not treat every situation the same. The league baseline gives a starting point for:
- expected run/pass balance
- expected formation usage
- expected tempo
- expected efficiency environment

That baseline is what makes later team-level tuning meaningful. Without it, every team risks behaving like the same generic CPU coach.

### Practical takeaway

This table helps define the **default situational behavior layer**.  
It tells a studio what the average NFL team tends to do in each situation before team-specific coaching identity is applied on top.

In [39]:
baseline_view.sort_values("dropback_rate_baseline", ascending=False)[
    ["situation_name", "dropback_rate_baseline", "rush_rate_baseline"]
]

,situation_name,dropback_rate_baseline,rush_rate_baseline
2,third_down,0.7494,0.2506
7,two_minute_half,0.7349,0.2651
8,two_minute_game,0.6818,0.3182
17,trailing_two_plus_scores,0.6770,0.3230
13,trailing,0.6309,0.3691
3,fourth_down,0.6225,0.3775
16,trailing_one_score,0.5968,0.4032
20,trailing_early_down,0.5899,0.4101
0,all_offense,0.5737,0.4263
9,one_score,0.5650,0.4350


In [40]:
baseline_view.sort_values("avg_epa_baseline", ascending=False)[
    ["situation_name", "avg_epa_baseline", "success_rate_baseline", "explosive_play_rate_baseline"]
]

,situation_name,avg_epa_baseline,success_rate_baseline,explosive_play_rate_baseline
3,fourth_down,0.1649,0.5443,0.1209
4,short_yardage,0.0647,0.5961,0.0906
15,leading_two_plus_scores,0.0204,0.4256,0.1375
12,leading,0.0162,0.4269,0.1357
14,leading_one_score,0.0136,0.4278,0.1347
10,neutral_early_down,0.0130,0.4391,0.1443
19,leading_early_down,0.0115,0.4235,0.1332
6,goal_to_go,0.0097,0.4380,0.0074
7,two_minute_half,0.0083,0.4231,0.1271
9,one_score,0.0035,0.4369,0.1332


## 5. Team Tendency Deviations
Which teams differ most from the league in the way they call games?

In [41]:
most_run_heavy = (
    team_baseline_features
    .sort_values("rush_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "rush_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_run_heavy

,team,situation_name,team_play_count,rush_rate_delta,avg_epa_delta,success_rate_delta
297,JAX,fourth_down,29,0.2777,-0.2789,0.0074
276,IND,fourth_down,27,0.2521,0.3578,0.1224
197,DEN,two_minute_game,34,0.1818,0.0153,-0.0118
302,JAX,two_minute_game,36,0.1818,-0.2651,-0.0331
527,PHI,fourth_down,20,0.1725,0.4490,0.0057
260,HOU,two_minute_game,45,0.1707,0.0737,-0.0164
532,PHI,two_minute_game,48,0.1610,-0.1998,-0.1025
36,ATL,leading_two_plus_scores,74,0.1590,-0.2159,-0.0337
518,NYJ,leading_one_score,63,0.1531,-0.3198,-0.1103
644,TEN,leading_two_plus_scores,23,0.1520,-0.4330,-0.0343


In [42]:
most_dropback_heavy = (
    team_baseline_features
    .sort_values("dropback_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "dropback_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_dropback_heavy

,team,situation_name,team_play_count,dropback_rate_delta,avg_epa_delta,success_rate_delta
8,ARI,two_minute_game,40,0.2432,-0.1142,-0.0192
653,WAS,fourth_down,26,0.2237,0.3523,0.0711
470,NO,two_minute_game,59,0.2165,0.0987,0.1312
5,ARI,red_zone,165,0.1880,-0.0802,-0.0345
553,PIT,two_minute_game,37,0.1831,-0.1011,-0.0428
384,LV,goal_to_go,40,0.1743,-0.1001,-0.1130
171,DAL,fourth_down,34,0.1716,0.4734,0.1028
132,CIN,goal_to_go,61,0.1636,-0.2580,-0.0282
321,KC,goal_to_go,78,0.1525,-0.2023,-0.0790
512,NYJ,two_minute_game,54,0.1515,-0.2088,-0.0794


### How to read this table

This table shows where teams differ most from the league baseline in run-pass tendency.

#### What the key numbers mean

- `rush_rate_delta`  
  Positive means the team is calling the run **more often than the league expectation** in that situation.  
  For CPU tuning, that suggests the team’s coach logic should skew more run-heavy there than a default league-average profile.

- `dropback_rate_delta`  
  Positive means the team is leaning more heavily into dropback behavior than baseline.  
  For CPU tuning, that suggests a more pass-oriented decision profile in that situation.

- `avg_epa_delta`  
  Positive means the team’s tendency in that situation is producing **better results than the league baseline**.  
  This matters because it suggests the behavior may be worth preserving in a team-specific tuning model.

- `success_rate_delta`  
  Positive means the team is sustaining offense more consistently than baseline in that situation.  
  That strengthens the case that the tendency is not random noise.

#### What counts as useful for tuning

The strongest tuning candidates are not just big tendency deviations. They are tendency deviations that are paired with:
- positive EPA
- positive success rate
- enough sample size to trust the pattern

#### Why this matters for CPU-controlled coaching decisions

This is where generic coach logic starts to separate into team identity.  
If a team is consistently more run-heavy or more dropback-heavy than baseline in a given situation, that can inform:
- playcall selection weights
- situation-specific aggression
- run/pass balance logic
- coaching personality tuning

### Practical takeaway

These tables help identify where a team should **not** behave like the league-average CPU coach.

In [44]:
most_no_huddle_heavy = (
    team_baseline_features
    .sort_values("no_huddle_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "no_huddle_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_no_huddle_heavy

,team,situation_name,team_play_count,no_huddle_rate_delta,avg_epa_delta,success_rate_delta
653,WAS,fourth_down,26,0.6749,0.3523,0.0711
652,WAS,third_down,187,0.5809,-0.1179,-0.0017
666,WAS,trailing_one_score,292,0.5596,0.0276,-0.0078
654,WAS,short_yardage,100,0.5458,-0.2425,-0.0461
659,WAS,one_score,551,0.5289,0.0291,0.0150
664,WAS,leading_one_score,123,0.5266,-0.0142,0.0031
660,WAS,neutral_early_down,352,0.5253,0.0322,0.0069
663,WAS,trailing,631,0.5179,0.0126,0.0089
650,WAS,all_offense,972,0.5165,0.0126,0.0223
669,WAS,leading_early_down,167,0.5099,0.0743,0.0496


## 6. Efficiency Deviations
Which teams are outperforming the baseline, and where?

In [45]:
most_efficient = (
    team_baseline_features
    .sort_values("avg_epa_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "avg_epa_delta",
            "success_rate_delta",
            "explosive_play_rate_delta",
        ]
    ]
)

most_efficient

,team,situation_name,team_play_count,avg_epa_delta,success_rate_delta,explosive_play_rate_delta
318,KC,fourth_down,32,1.3826,0.2057,0.0354
87,CAR,fourth_down,39,1.1085,0.1480,0.1612
444,NE,fourth_down,25,1.0384,0.1757,0.0791
129,CIN,fourth_down,20,0.8741,0.1557,0.0291
339,LA,fourth_down,30,0.8632,0.1557,0.0458
45,BAL,fourth_down,25,0.5297,0.0557,0.0391
66,BUF,fourth_down,32,0.5230,0.0495,0.0666
171,DAL,fourth_down,34,0.4734,0.1028,0.0262
458,NE,trailing_two_plus_scores,39,0.4699,0.1317,0.0252
527,PHI,fourth_down,20,0.4490,0.0057,-0.0209


### How to read this table

This table highlights the team-situation combinations that outperform the league baseline the most.

#### What the key numbers mean

- `avg_epa_delta`  
  Positive means the team is generating more value per play than the league baseline in that situation.  
  For tuning, this suggests a behavior pattern that is not just distinct, but effective.

- `success_rate_delta`  
  Positive means the team is staying ahead of schedule more often than the baseline.  
  That makes the behavior more trustworthy as a tuning signal.

- `explosive_play_rate_delta`  
  Positive means the team creates more chunk plays than baseline in that situation.  
  This helps show whether the offense creates pressure on a defense in a distinct way.

#### Why this matters for CPU-controlled coaching decisions

A studio should be careful not to tune CPU coaches only around style.  
Some styles are different, but not better.  
This table highlights where the team’s situational identity is actually driving results above the baseline.

That makes these situations stronger candidates for:
- preserving team personality
- weighting certain play families more heavily
- tuning aggression or tempo in a way that reflects real behavior

### Practical takeaway

This table helps separate **distinctive behavior** from **distinctive and effective behavior**, which is much more valuable for CPU tuning.

In [46]:
least_efficient = (
    team_baseline_features
    .sort_values("avg_epa_delta", ascending=True)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "avg_epa_delta",
            "success_rate_delta",
            "explosive_play_rate_delta",
        ]
    ]
)

least_efficient

,team,situation_name,team_play_count,avg_epa_delta,success_rate_delta,explosive_play_rate_delta
465,NO,fourth_down,34,-1.0421,-0.1325,-0.1209
150,CLE,fourth_down,29,-0.9231,-0.1305,-0.0864
381,LV,fourth_down,32,-0.8048,-0.1068,-0.0584
611,TB,fourth_down,29,-0.5899,-0.0615,-0.0864
489,NYG,goal_to_go,70,-0.5679,-0.1666,-0.0074
402,MIA,fourth_down,23,-0.5260,-0.1095,-0.0774
473,NO,tied,122,-0.4429,-0.1072,-0.0159
24,ATL,fourth_down,28,-0.4381,-0.0443,-0.0852
644,TEN,leading_two_plus_scores,23,-0.4330,-0.0343,0.0364
392,LV,leading_one_score,98,-0.4114,-0.1115,-0.0020


In [47]:
most_successful = (
    team_baseline_features
    .sort_values("success_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "team_play_count",
            "success_rate_delta",
            "avg_epa_delta",
            "explosive_play_rate_delta",
        ]
    ]
)

most_successful

,team,situation_name,team_play_count,success_rate_delta,avg_epa_delta,explosive_play_rate_delta
318,KC,fourth_down,32,0.2057,1.3826,0.0354
444,NE,fourth_down,25,0.1757,1.0384,0.0791
237,GB,goal_to_go,54,0.1731,0.1858,-0.0074
339,LA,fourth_down,30,0.1557,0.8632,0.0458
129,CIN,fourth_down,20,0.1557,0.8741,0.0291
87,CAR,fourth_down,39,0.1480,1.1085,0.1612
365,LAC,two_minute_game,47,0.1377,0.1933,0.0009
342,LA,goal_to_go,91,0.1334,0.3020,-0.0074
458,NE,trailing_two_plus_scores,39,0.1317,0.4699,0.0252
470,NO,two_minute_game,59,0.1312,0.0987,0.0257


## 8. Team Deep Dive
Use one team code at a time to study how that team differs from league baseline across situations.

In [50]:
TEAM_CODE = "BUF"

In [51]:
team_view = (
    team_baseline_features
    .loc[team_baseline_features["team"] == TEAM_CODE]
    .sort_values("situation_order")
)

team_view[
    [
        "team",
        "situation_name",
        "team_play_count",
        "dropback_rate_delta",
        "rush_rate_delta",
        "shotgun_rate_delta",
        "no_huddle_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
        "explosive_play_rate_delta",
    ]
]

,team,situation_name,team_play_count,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta
63,BUF,all_offense,1056,-0.0680,0.0680,-0.1948,-0.0423,0.1464,0.0532,0.0263
64,BUF,early_down,819,-0.0679,0.0679,-0.2138,-0.0451,0.1182,0.0521,0.0240
65,BUF,third_down,205,-0.0762,0.0762,-0.1197,-0.0290,0.1919,0.0538,0.0303
66,BUF,fourth_down,32,0.0650,-0.0650,-0.0812,-0.0630,0.5230,0.0495,0.0666
67,BUF,short_yardage,128,-0.0429,0.0429,-0.2041,0.0233,0.0047,0.0289,0.0032
68,BUF,red_zone,190,-0.0871,0.0871,-0.2484,0.0529,0.1341,0.0723,-0.0091
69,BUF,goal_to_go,76,-0.1468,0.1468,-0.3347,0.1376,0.1853,0.1015,-0.0074
70,BUF,two_minute_half,91,-0.1415,0.1415,-0.2404,-0.0535,0.1188,0.0384,0.0158
71,BUF,two_minute_game,51,-0.1328,0.1328,-0.1898,-0.0157,0.1411,0.0568,0.0105
72,BUF,one_score,649,-0.0565,0.0565,-0.1728,-0.0370,0.1083,0.0454,0.0348


### How to read this table

This section shows how one team differs from league baseline across every major situation. This is one of the clearest tables for thinking about CPU-controlled coaching DNA.

#### What the key numbers mean

- `dropback_rate_delta`  
  Positive means the team should behave more pass-oriented than the default CPU baseline in that situation.  
  Negative means it should behave more run-oriented.

- `shotgun_rate_delta`  
  Positive means the team should use shotgun more often than the default baseline in that situation.  
  This can inform formation tuning.

- `no_huddle_rate_delta`  
  Positive means the team should use faster tempo than the baseline.  
  This can inform hurry-up and pace logic.

- `avg_epa_delta` and `success_rate_delta`  
  Positive means the team’s behavior is outperforming the league norm.  
  That makes the tendency more credible as a design input.

- `explosive_play_rate_delta`  
  Positive means the team creates chunk plays more often than baseline.  
  This can influence how aggressive or vertical the CPU behavior should feel.

#### Why this matters for CPU-controlled coaching decisions

This is where the project starts to resemble a tuning guide.  
If a team consistently differs from baseline in a specific situation and those differences are productive, that suggests the CPU should:
- call plays differently
- pace drives differently
- lean into different structures
- feel more like that specific coaching identity instead of a generic average team

### Practical takeaway

This table helps answer:  
**How should this team’s CPU coach behave differently from a league-average default in each game situation?**

## 9. Score Sanity Checks
These checks help verify that the final scores broadly line up with the underlying deltas.

In [54]:
coach_dna_team_summary.sort_values("overall_coach_dna_score", ascending=False).head(15)

,profile_season,team,scored_situations,overall_coach_dna_score,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_situation,top_signal_score,lowest_signal_situation,lowest_signal_score
0,2025,LA,13,71.4506,62.3970,88.1404,70.5575,71.6600,96.3510,neutral_early_down,83.9844,two_minute_game,34.2305
1,2025,BUF,13,70.8384,72.2995,77.7592,70.5647,48.6145,96.0830,third_down,79.5703,fourth_down,45.8760
2,2025,WAS,13,68.2717,80.9874,56.2046,64.8656,45.0901,96.2908,tied_early_down,82.3633,two_minute_game,37.3359
3,2025,CIN,13,65.4883,74.2507,65.0623,41.7981,56.2964,97.3112,red_zone,82.0703,fourth_down,51.8379
4,2025,BAL,13,61.8666,70.4730,54.6368,68.3142,27.7396,95.3569,leading_early_down,70.8594,two_minute_game,22.9512
5,2025,CHI,13,61.8290,53.5188,66.4914,65.3718,71.8608,97.1988,third_down,78.3203,fourth_down,35.2266
6,2025,NE,13,59.1905,48.3904,77.7351,70.7027,36.2316,96.4879,two_minute_half,79.9609,short_yardage,41.5234
7,2025,SEA,13,59.0944,58.3097,59.7254,63.4167,45.3474,96.3373,trailing_one_score,76.9531,two_minute_game,15.0264
8,2025,GB,13,57.8559,47.3359,67.0709,70.3521,55.6716,96.1749,third_down,83.2422,tied_early_down,25.4297
9,2025,SF,13,57.1461,48.3846,72.8378,45.9918,63.2382,96.6462,third_down,66.8945,fourth_down,33.5566


In [55]:
coach_dna_situation_scores.sort_values("coach_dna_score_adjusted", ascending=False).head(20)[
    [
        "team",
        "situation_name",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_signal_score",
        "efficiency_signal_score",
        "explosiveness_signal_score",
        "stability_signal_score",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]
]

,team,situation_name,team_play_count,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,tendency_profile_label,efficiency_profile_label
661,WAS,tied,136,86.953125,95.31250,78.125000,86.458333,65.62500,more_run_heavy_than_baseline,more_efficient_than_baseline
63,BUF,all_offense,1056,84.296875,82.03125,94.531250,91.666667,50.00000,more_run_heavy_than_baseline,more_efficient_than_baseline
346,LA,neutral_early_down,448,83.984375,74.21875,97.656250,92.708333,72.65625,more_dropback_heavy_than_baseline,more_efficient_than_baseline
233,GB,third_down,206,83.242188,72.65625,92.187500,95.833333,81.25000,more_run_heavy_than_baseline,more_efficient_than_baseline
345,LA,one_score,739,82.734375,71.87500,100.000000,85.416667,75.78125,more_dropback_heavy_than_baseline,more_efficient_than_baseline
668,WAS,tied_early_down,104,82.363281,95.31250,66.015625,86.458333,50.00000,more_run_heavy_than_baseline,close_to_baseline_efficiency
131,CIN,red_zone,146,82.070312,82.03125,90.625000,73.437500,64.84375,more_dropback_heavy_than_baseline,more_efficient_than_baseline
64,BUF,early_down,819,82.031250,80.46875,89.843750,84.895833,56.25000,more_run_heavy_than_baseline,more_efficient_than_baseline
80,BUF,trailing_two_plus_scores,234,81.914062,81.25000,86.718750,77.083333,71.09375,more_run_heavy_than_baseline,more_efficient_than_baseline
340,LA,short_yardage,129,81.875000,73.43750,89.062500,84.375000,89.06250,more_dropback_heavy_than_baseline,more_efficient_than_baseline


In [56]:
coach_dna_situation_scores.sort_values("coach_dna_score_adjusted", ascending=True).head(20)[
    [
        "team",
        "situation_name",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_signal_score",
        "efficiency_signal_score",
        "explosiveness_signal_score",
        "stability_signal_score",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]
]

,team,situation_name,team_play_count,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,tendency_profile_label,efficiency_profile_label
574,SEA,two_minute_game,37,15.026367,20.312500,17.578125,16.666667,12.50000,close_to_baseline_run_pass_split,less_efficient_than_baseline
611,TB,fourth_down,29,17.355469,12.890625,23.046875,25.000000,50.78125,close_to_baseline_run_pass_split,less_efficient_than_baseline
509,NYJ,red_zone,121,17.500000,7.031250,11.718750,18.750000,35.93750,close_to_baseline_run_pass_split,less_efficient_than_baseline
381,LV,fourth_down,32,20.050781,25.781250,14.843750,30.208333,41.40625,close_to_baseline_run_pass_split,less_efficient_than_baseline
153,CLE,goal_to_go,54,20.826563,11.718750,13.281250,39.062500,46.87500,close_to_baseline_run_pass_split,less_efficient_than_baseline
465,NO,fourth_down,34,21.295898,48.437500,5.078125,11.979167,7.81250,close_to_baseline_run_pass_split,less_efficient_than_baseline
24,ATL,fourth_down,28,21.442383,23.437500,30.859375,19.791667,46.09375,close_to_baseline_run_pass_split,less_efficient_than_baseline
147,CLE,all_offense,1024,22.773438,26.562500,3.906250,10.416667,32.81250,close_to_baseline_run_pass_split,less_efficient_than_baseline
638,TEN,one_score,558,22.949219,17.187500,7.421875,27.604167,42.18750,close_to_baseline_run_pass_split,less_efficient_than_baseline
50,BAL,two_minute_game,38,22.951172,17.968750,32.812500,52.083333,37.50000,close_to_baseline_run_pass_split,close_to_baseline_efficiency


## 10. Draft Findings
Write down early insights worth carrying into the README, GitHub post, or interview narrative.

### Draft findings
- 
- 
- 
- 
- 